In [6]:
import pandas as pd

# Use the correct separator "|"
data_path = r"D:\Personal\KAIM-10 Academy\Week 3\Project Work\Insurance Analytics_Predictive Modeling-Week 3\data\raw\raw_data.txt"

df = pd.read_csv(data_path, sep="|")  # pipe-separated
print("Shape:", df.shape)
print("Columns:", df.columns.tolist())


C:\Users\user\AppData\Local\Temp\ipykernel_29008\680328912.py:6: DtypeWarning: Columns (32,37) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv(data_path, sep="|")  # pipe-separated


Shape: (1000098, 52)
Columns: ['UnderwrittenCoverID', 'PolicyID', 'TransactionMonth', 'IsVATRegistered', 'Citizenship', 'LegalType', 'Title', 'Language', 'Bank', 'AccountType', 'MaritalStatus', 'Gender', 'Country', 'Province', 'PostalCode', 'MainCrestaZone', 'SubCrestaZone', 'ItemType', 'mmcode', 'VehicleType', 'RegistrationYear', 'make', 'Model', 'Cylinders', 'cubiccapacity', 'kilowatts', 'bodytype', 'NumberOfDoors', 'VehicleIntroDate', 'CustomValueEstimate', 'AlarmImmobiliser', 'TrackingDevice', 'CapitalOutstanding', 'NewVehicle', 'WrittenOff', 'Rebuilt', 'Converted', 'CrossBorder', 'NumberOfVehiclesInFleet', 'SumInsured', 'TermFrequency', 'CalculatedPremiumPerTerm', 'ExcessSelected', 'CoverCategory', 'CoverType', 'CoverGroup', 'Section', 'Product', 'StatutoryClass', 'StatutoryRiskType', 'TotalPremium', 'TotalClaims']


In [7]:
# Check missing values
missing = df.isnull().sum().sort_values(ascending=False)
print(missing.head(20))


NumberOfVehiclesInFleet    1000098
CrossBorder                 999400
CustomValueEstimate         779642
Rebuilt                     641901
Converted                   641901
WrittenOff                  641901
NewVehicle                  153295
Bank                        145961
AccountType                  40232
Gender                        9536
MaritalStatus                 8259
VehicleType                    552
make                           552
mmcode                         552
Model                          552
Cylinders                      552
bodytype                       552
kilowatts                      552
NumberOfDoors                  552
VehicleIntroDate               552
dtype: int64


In [8]:
# List all columns to see exact names
print(df.columns.tolist())

# Drop safely using errors='ignore'
df.drop(columns=[
    'NumberOfVehiclesInFleet',
    'CrossBorder',
    'CustomValueEstimate',
    'Rebuilt',
    'Converted',
    'WrittenOff'
], inplace=True, errors='ignore')


['UnderwrittenCoverID', 'PolicyID', 'TransactionMonth', 'IsVATRegistered', 'Citizenship', 'LegalType', 'Title', 'Language', 'Bank', 'AccountType', 'MaritalStatus', 'Gender', 'Country', 'Province', 'PostalCode', 'MainCrestaZone', 'SubCrestaZone', 'ItemType', 'mmcode', 'VehicleType', 'RegistrationYear', 'make', 'Model', 'Cylinders', 'cubiccapacity', 'kilowatts', 'bodytype', 'NumberOfDoors', 'VehicleIntroDate', 'CustomValueEstimate', 'AlarmImmobiliser', 'TrackingDevice', 'CapitalOutstanding', 'NewVehicle', 'WrittenOff', 'Rebuilt', 'Converted', 'CrossBorder', 'NumberOfVehiclesInFleet', 'SumInsured', 'TermFrequency', 'CalculatedPremiumPerTerm', 'ExcessSelected', 'CoverCategory', 'CoverType', 'CoverGroup', 'Section', 'Product', 'StatutoryClass', 'StatutoryRiskType', 'TotalPremium', 'TotalClaims']


In [19]:
# Convert 'TransactionMonth' and 'VehicleIntroDate' to datetime
df['TransactionMonth'] = pd.to_datetime(df['TransactionMonth'], errors='coerce')
df['VehicleIntroDate'] = pd.to_datetime(df['VehicleIntroDate'], errors='coerce')

# Extract year/month if useful (already done as VehicleIntroYear/Month)
df['VehicleIntroYear'] = df['VehicleIntroDate'].dt.year
df['VehicleIntroMonth'] = df['VehicleIntroDate'].dt.month


In [20]:
# Target column
target = 'TotalClaims'  # or 'claim_severity' if available in full data

# Features
X = df.drop(columns=[target], errors='ignore')  # ignore if target not in df
y = df[target] if target in df.columns else None

# Identify numerical vs categorical columns
num_cols = X.select_dtypes(include=['int64', 'float64']).columns.tolist()
cat_cols = X.select_dtypes(include=['object', 'bool']).columns.tolist()


In [11]:
from sklearn.preprocessing import OneHotEncoder
from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline

# Categorical imputer + encoder pipeline
cat_imputer = SimpleImputer(strategy='most_frequent')
cat_encoder = OneHotEncoder(handle_unknown='ignore', sparse_output=False)

cat_pipeline = Pipeline(steps=[
    ('impute', cat_imputer),
    ('encode', cat_encoder)
])

In [21]:
from sklearn.preprocessing import OneHotEncoder
from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline

# Categorical imputer + encoder pipeline
cat_imputer = SimpleImputer(strategy='most_frequent')
cat_encoder = OneHotEncoder(handle_unknown='ignore', sparse_output=False)

cat_pipeline = Pipeline(steps=[
    ('impute', cat_imputer),
    ('encode', cat_encoder)
])

In [13]:
from sklearn.compose import ColumnTransformer

preprocessor = ColumnTransformer([
    ('num', 'passthrough', num_cols),        # numeric columns: just passthrough for now
    ('cat', cat_pipeline, cat_cols)          # categorical pipeline
])

In [14]:
# Ensure all categorical columns are string type
for col in cat_cols:
    df[col] = df[col].astype(str)

In [15]:
X = df[num_cols + cat_cols]  # numeric + categorical
X_processed = preprocessor.fit_transform(X)
print("Processed features shape:", X_processed.shape)

Processed features shape: (1000098, 1865)


In [27]:
from sklearn.preprocessing import LabelEncoder

# Identify categorical columns
cat_cols = df.select_dtypes(include=['object', 'category']).columns.tolist()

# Split low vs high cardinality
low_card_cols = [c for c in cat_cols if df[c].nunique() <= 20]
high_card_cols = [c for c in cat_cols if df[c].nunique() > 20]

print("Low-cardinality:", low_card_cols)
print("High-cardinality:", high_card_cols)

# ----- One-hot encode low-cardinality -----
df = pd.get_dummies(df, columns=low_card_cols, drop_first=True)

# ----- Label encode high-cardinality -----
le = LabelEncoder()
for col in high_card_cols:
    df[col] = le.fit_transform(df[col])

print("After encoding:", df.shape)

Low-cardinality: []
High-cardinality: []
After encoding: (1000098, 134)


In [24]:
df.columns.tolist()

['UnderwrittenCoverID',
 'PolicyID',
 'TransactionMonth',
 'PostalCode',
 'SubCrestaZone',
 'mmcode',
 'RegistrationYear',
 'make',
 'Model',
 'Cylinders',
 'cubiccapacity',
 'kilowatts',
 'NumberOfDoors',
 'VehicleIntroDate',
 'CapitalOutstanding',
 'SumInsured',
 'CalculatedPremiumPerTerm',
 'CoverCategory',
 'CoverType',
 'TotalPremium',
 'TotalClaims',
 'VehicleIntroYear',
 'VehicleIntroMonth',
 'IsVATRegistered_True',
 'Citizenship_AF',
 'Citizenship_ZA',
 'Citizenship_ZW',
 'LegalType_Individual',
 'LegalType_Partnership',
 'LegalType_Private company',
 'LegalType_Public company',
 'LegalType_Sole proprieter',
 'Title_Miss',
 'Title_Mr',
 'Title_Mrs',
 'Title_Ms',
 'Bank_Capitec Bank',
 'Bank_First National Bank',
 'Bank_FirstRand Bank',
 'Bank_Investec Bank',
 'Bank_Ithala Bank',
 'Bank_Mercantile Lisbon Bank',
 'Bank_Nedbank',
 'Bank_Old Mutual',
 'Bank_RMB Private Bank',
 'Bank_Standard Bank',
 'Bank_nan',
 'AccountType_Savings account',
 'AccountType_Transmission account',
 '

In [34]:
import pandas as pd

data_path = "../data/processed/cleaned_data.csv"

# Skip bad lines to avoid the parser error
df = pd.read_csv(data_path, low_memory=False, on_bad_lines='skip')
print("Loaded shape:", df.shape)


Loaded shape: (1000098, 55)


In [36]:
# Identify numeric columns
num_cols = df.select_dtypes(include=['int64', 'float64']).columns.tolist()

# Drop numeric columns with 100% missing
full_missing_nums = [c for c in num_cols if df[c].isnull().sum() == len(df)]
if full_missing_nums:
    print("Dropping numeric columns with 100% missing:", full_missing_nums)
    df.drop(columns=full_missing_nums, inplace=True)
    # Update num_cols
    num_cols = [c for c in num_cols if c not in full_missing_nums]

# Impute remaining numeric columns
from sklearn.impute import SimpleImputer
num_imputer = SimpleImputer(strategy='median')
df[num_cols] = num_imputer.fit_transform(df[num_cols])

Dropping numeric columns with 100% missing: ['numberofvehiclesinfleet']


In [37]:
cat_cols = df.select_dtypes(include=['object', 'category']).columns.tolist()
cat_imputer = SimpleImputer(strategy='most_frequent')
df[cat_cols] = cat_imputer.fit_transform(df[cat_cols])

In [42]:
# ================== 1️⃣ Load & clean data ==================
import pandas as pd
import numpy as np
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import LabelEncoder
from sklearn.model_selection import train_test_split

data_path = "../data/processed/cleaned_data.csv"

# Load CSV, skip bad lines
df = pd.read_csv(data_path, low_memory=False, on_bad_lines='skip')
print("Loaded shape:", df.shape)

# ---------------- Numeric Imputation ----------------
num_cols = df.select_dtypes(include=['int64', 'float64']).columns.tolist()
# Drop fully empty numeric columns
full_missing_nums = [c for c in num_cols if df[c].isnull().sum() == len(df)]
if full_missing_nums:
    print("Dropping numeric columns with 100% missing:", full_missing_nums)
    df.drop(columns=full_missing_nums, inplace=True)
    num_cols = [c for c in num_cols if c not in full_missing_nums]

num_imputer = SimpleImputer(strategy='median')
df[num_cols] = num_imputer.fit_transform(df[num_cols])

# ---------------- Categorical Imputation ----------------
cat_cols = df.select_dtypes(include=['object', 'category']).columns.tolist()
cat_imputer = SimpleImputer(strategy='most_frequent')
df[cat_cols] = cat_imputer.fit_transform(df[cat_cols])

# ---------------- Encoding ----------------
for c in cat_cols:
    df[c] = LabelEncoder().fit_transform(df[c])

print("After cleaning & encoding:", df.shape)


Loaded shape: (1000098, 55)
Dropping numeric columns with 100% missing: ['numberofvehiclesinfleet']
After cleaning & encoding: (1000098, 54)


In [39]:
# List all columns
print(df.columns.tolist())

['underwrittencoverid', 'policyid', 'transactionmonth', 'isvatregistered', 'citizenship', 'legaltype', 'title', 'language', 'bank', 'accounttype', 'maritalstatus', 'gender', 'country', 'province', 'postalcode', 'maincrestazone', 'subcrestazone', 'itemtype', 'mmcode', 'vehicletype', 'registrationyear', 'make', 'model', 'cylinders', 'cubiccapacity', 'kilowatts', 'bodytype', 'numberofdoors', 'vehicleintrodate', 'customvalueestimate', 'alarmimmobiliser', 'trackingdevice', 'capitaloutstanding', 'newvehicle', 'writtenoff', 'rebuilt', 'converted', 'crossborder', 'suminsured', 'termfrequency', 'calculatedpremiumperterm', 'excessselected', 'covercategory', 'covertype', 'covergroup', 'section', 'product', 'statutoryclass', 'statutoryrisktype', 'totalpremium', 'totalclaims', 'claim_occurred', 'claim_severity', 'margin']


In [40]:
# Correct target columns
target_reg = 'calculatedpremiumperterm'
target_cls = 'claim_occurred'

# Features
X = df.drop([target_reg, target_cls], axis=1)
y_reg = df[target_reg]
y_cls = df[target_cls]

# Train/test split
from sklearn.model_selection import train_test_split

X_train_reg, X_test_reg, y_train_reg, y_test_reg = train_test_split(
    X, y_reg, test_size=0.3, random_state=42
)
X_train_cls, X_test_cls, y_train_cls, y_test_cls = train_test_split(
    X, y_cls, test_size=0.3, random_state=42, stratify=y_cls
)

print("Regression Train:", X_train_reg.shape, "Test:", X_test_reg.shape)
print("Classification Train:", X_train_cls.shape, "Test:", X_test_cls.shape)

Regression Train: (700068, 52) Test: (300030, 52)
Classification Train: (700068, 52) Test: (300030, 52)


In [41]:
from sklearn.ensemble import RandomForestRegressor, RandomForestClassifier
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score, accuracy_score, precision_score, recall_score, f1_score, roc_auc_score
import numpy as np

# ----------------------------
# Random Forest Regression
# ----------------------------
rf_reg = RandomForestRegressor(
    n_estimators=50,  # fewer trees for speed
    max_depth=8,
    random_state=42,
    n_jobs=-1
)
rf_reg.fit(X_train_reg, y_train_reg)
y_pred_rf_reg = rf_reg.predict(X_test_reg)

print("Random Forest Regression Metrics:")
print("MAE:", mean_absolute_error(y_test_reg, y_pred_rf_reg))
print("RMSE:", np.sqrt(mean_squared_error(y_test_reg, y_pred_rf_reg)))
print("R2:", r2_score(y_test_reg, y_pred_rf_reg))

# ----------------------------
# Random Forest Classification
# ----------------------------
rf_cls = RandomForestClassifier(
    n_estimators=50,
    max_depth=8,
    random_state=42,
    n_jobs=-1,
    class_weight='balanced'
)
rf_cls.fit(X_train_cls, y_train_cls)
y_pred_rf_cls = rf_cls.predict(X_test_cls)
y_pred_rf_prob = rf_cls.predict_proba(X_test_cls)[:,1]

print("\nRandom Forest Classification Metrics:")
print("Accuracy:", accuracy_score(y_test_cls, y_pred_rf_cls))
print("Precision:", precision_score(y_test_cls, y_pred_rf_cls))
print("Recall:", recall_score(y_test_cls, y_pred_rf_cls))
print("F1 Score:", f1_score(y_test_cls, y_pred_rf_cls))
print("ROC-AUC:", roc_auc_score(y_test_cls, y_pred_rf_prob))

Random Forest Regression Metrics:
MAE: 11.2970732170493
RMSE: 40.498742047690946
R2: 0.9886577580850746

Random Forest Classification Metrics:
Accuracy: 1.0
Precision: 1.0
Recall: 1.0
F1 Score: 1.0
ROC-AUC: 0.9999999999999999


In [13]:
import pandas as pd

data_path = r"D:\Personal\KAIM-10 Academy\Week 3\Project Work\Insurance Analytics_Predictive Modeling-Week 3\data\processed\insurance_data_cleaned.csv"

df = pd.read_csv(data_path, low_memory=False)
print("Data shape:", df.shape)
print("Columns:", df.columns.tolist())

Data shape: (1000098, 52)
Columns: ['UnderwrittenCoverID', 'PolicyID', 'TransactionMonth', 'IsVATRegistered', 'Citizenship', 'LegalType', 'Title', 'Language', 'Bank', 'AccountType', 'MaritalStatus', 'Gender', 'Country', 'Province', 'PostalCode', 'MainCrestaZone', 'SubCrestaZone', 'ItemType', 'mmcode', 'VehicleType', 'RegistrationYear', 'make', 'Model', 'Cylinders', 'cubiccapacity', 'kilowatts', 'bodytype', 'NumberOfDoors', 'VehicleIntroDate', 'CustomValueEstimate', 'AlarmImmobiliser', 'TrackingDevice', 'CapitalOutstanding', 'NewVehicle', 'WrittenOff', 'Rebuilt', 'Converted', 'CrossBorder', 'SumInsured', 'TermFrequency', 'CalculatedPremiumPerTerm', 'ExcessSelected', 'CoverCategory', 'CoverType', 'CoverGroup', 'Section', 'Product', 'StatutoryClass', 'StatutoryRiskType', 'TotalPremium', 'TotalClaims', 'LossRatio']


In [17]:
from sklearn.preprocessing import LabelEncoder

cat_cols = df.select_dtypes(include=['object', 'category']).columns.tolist()
le = LabelEncoder()
for col in cat_cols:
    df[col] = le.fit_transform(df[col].astype(str))  # convert to string first

In [19]:
print(df.columns.tolist())

['UnderwrittenCoverID', 'PolicyID', 'TransactionMonth', 'IsVATRegistered', 'Citizenship', 'LegalType', 'Title', 'Language', 'Bank', 'AccountType', 'MaritalStatus', 'Gender', 'Country', 'Province', 'PostalCode', 'MainCrestaZone', 'SubCrestaZone', 'ItemType', 'mmcode', 'VehicleType', 'RegistrationYear', 'make', 'Model', 'Cylinders', 'cubiccapacity', 'kilowatts', 'bodytype', 'NumberOfDoors', 'VehicleIntroDate', 'CustomValueEstimate', 'AlarmImmobiliser', 'TrackingDevice', 'CapitalOutstanding', 'NewVehicle', 'WrittenOff', 'Rebuilt', 'Converted', 'CrossBorder', 'SumInsured', 'TermFrequency', 'CalculatedPremiumPerTerm', 'ExcessSelected', 'CoverCategory', 'CoverType', 'CoverGroup', 'Section', 'Product', 'StatutoryClass', 'StatutoryRiskType', 'TotalPremium', 'TotalClaims', 'LossRatio']


In [20]:
from sklearn.model_selection import train_test_split

# ----------------------------
# Set targets
# ----------------------------
target_reg = 'CalculatedPremiumPerTerm'
# For classification, choose a binary target if available, e.g. 'LossRatio' thresholded
# Here, let's create a dummy binary target for illustration
import numpy as np
df['ClaimOccurred'] = (df['LossRatio'] > 0).astype(int)
target_cls = 'ClaimOccurred'

# ----------------------------
# Features
# ----------------------------
X = df.drop([target_reg, target_cls], axis=1)
y_reg = df[target_reg]
y_cls = df[target_cls]

# ----------------------------
# Train-test split
# ----------------------------
X_train_reg, X_test_reg, y_train_reg, y_test_reg = train_test_split(
    X, y_reg, test_size=0.3, random_state=42
)

X_train_cls, X_test_cls, y_train_cls, y_test_cls = train_test_split(
    X, y_cls, test_size=0.3, random_state=42
)

print("Regression X_train shape:", X_train_reg.shape)
print("Classification X_train shape:", X_train_cls.shape)

Regression X_train shape: (700068, 51)
Classification X_train shape: (700068, 51)


In [4]:
import pandas as pd

# Regression
X_train_reg = pd.read_csv(r"D:\Personal\KAIM-10 Academy\Week 3\Project Work\Insurance Analytics_Predictive Modeling-Week 3\data\processed\X_train_regression.csv")
X_test_reg = pd.read_csv(r"D:\Personal\KAIM-10 Academy\Week 3\Project Work\Insurance Analytics_Predictive Modeling-Week 3\data\processed\X_test_regression.csv")
y_train_reg = pd.read_csv(r"D:\Personal\KAIM-10 Academy\Week 3\Project Work\Insurance Analytics_Predictive Modeling-Week 3\data\processed\y_train_regression.csv").squeeze()
y_test_reg = pd.read_csv(r"D:\Personal\KAIM-10 Academy\Week 3\Project Work\Insurance Analytics_Predictive Modeling-Week 3\data\processed\y_test_regression.csv").squeeze()

# Classification
X_train_cls = pd.read_csv(r"D:\Personal\KAIM-10 Academy\Week 3\Project Work\Insurance Analytics_Predictive Modeling-Week 3\data\processed\X_train_classification.csv")
X_test_cls = pd.read_csv(r"D:\Personal\KAIM-10 Academy\Week 3\Project Work\Insurance Analytics_Predictive Modeling-Week 3\data\processed\X_test_classification.csv")
y_train_cls = pd.read_csv(r"D:\Personal\KAIM-10 Academy\Week 3\Project Work\Insurance Analytics_Predictive Modeling-Week 3\data\processed\y_train_classification.csv").squeeze()
y_test_cls = pd.read_csv(r"D:\Personal\KAIM-10 Academy\Week 3\Project Work\Insurance Analytics_Predictive Modeling-Week 3\data\processed\y_test_classification.csv").squeeze()

In [5]:
print("Regression X_train shape:", X_train_reg.shape)
print("Regression y_train shape:", y_train_reg.shape)
print("Classification X_train shape:", X_train_cls.shape)
print("Classification y_train shape:", y_train_cls.shape)

Regression X_train shape: (487358, 211)
Regression y_train shape: (700068,)
Classification X_train shape: (700068, 211)
Classification y_train shape: (700068,)


In [7]:
print(X_train_reg.shape)
print(y_train_reg.shape)
print(X_test_reg.shape)
print(y_test_reg.shape)

(487358, 211)
(700068,)
(300030, 211)
(300030,)


In [1]:
import pandas as pd

X_train_reg = pd.read_csv("../data/processed/X_train_regression.csv")
X_test_reg = pd.read_csv("../data/processed/X_test_regression.csv")
y_train_reg = pd.read_csv("../data/processed/y_train_regression.csv").squeeze()
y_test_reg = pd.read_csv("../data/processed/y_test_regression.csv").squeeze()

print("Regression X_train shape:", X_train_reg.shape)
print("Regression y_train shape:", y_train_reg.shape)


Regression X_train shape: (487358, 211)
Regression y_train shape: (700068,)


In [4]:
from sklearn.preprocessing import OneHotEncoder

# Identify categorical columns
categorical_cols = X_train_reg.select_dtypes(include='object').columns

# One-Hot Encode
X_train_reg_encoded = pd.get_dummies(X_train_reg, columns=categorical_cols, drop_first=True)
X_test_reg_encoded = pd.get_dummies(X_test_reg, columns=categorical_cols, drop_first=True)

# Align columns in case some categories are missing in test
X_test_reg_encoded = X_test_reg_encoded.reindex(columns=X_train_reg_encoded.columns, fill_value=0)


In [6]:
print("X_train_reg rows:", X_train_reg.shape[0])
print("y_train_reg rows:", y_train_reg.shape[0])


X_train_reg rows: 487358
y_train_reg rows: 700068


In [7]:
# Ensure X and y have the same number of rows
y_train_reg_aligned = y_train_reg[:X_train_reg.shape[0]]

print("X_train_reg rows:", X_train_reg.shape[0])
print("y_train_reg_aligned rows:", y_train_reg_aligned.shape[0])

X_train_reg rows: 487358
y_train_reg_aligned rows: 487358


In [9]:
# List columns that are not numeric
non_numeric_cols = X_train_reg.select_dtypes(exclude=[np.number]).columns
print("Non-numeric columns:", non_numeric_cols)


Non-numeric columns: Index(['isvatregistered', 'alarmimmobiliser', 'trackingdevice',
       'engine_size_cat', 'transactionmonth_2013-11-01',
       'transactionmonth_2013-12-01', 'transactionmonth_2014-01-01',
       'transactionmonth_2014-02-01', 'transactionmonth_2014-03-01',
       'transactionmonth_2014-04-01',
       ...
       'covergroup_Standalone passenger liability',
       'covergroup_Third Party Only', 'covergroup_Trailer',
       'section_Motor Comprehensive', 'section_Optional Extended Covers',
       'section_Standalone passenger liability',
       'section_Third party or third party, fire and theft only',
       'product_Mobility Commercial Cover: Monthly',
       'product_Mobility Metered Taxis: Monthly',
       'product_Standalone Passenger Liability'],
      dtype='object', length=189)


In [3]:
import pandas as pd
from sklearn.ensemble import RandomForestRegressor

# Load regression data
X_train_reg = pd.read_csv("../data/processed/X_train_regression.csv")
X_test_reg = pd.read_csv("../data/processed/X_test_regression.csv")
y_train_reg = pd.read_csv("../data/processed/y_train_regression.csv").squeeze()

# Align y_train in case it's longer than X_train
y_train_reg_aligned = y_train_reg.iloc[:len(X_train_reg)]

# One-hot encode categorical columns
X_train_encoded = pd.get_dummies(X_train_reg, drop_first=True)
X_test_encoded = pd.get_dummies(X_test_reg, drop_first=True)

# Align columns between train and test
X_test_encoded = X_test_encoded.reindex(columns=X_train_encoded.columns, fill_value=0)

# Take a small subset to test quickly
X_train_small = X_train_encoded.sample(n=5000, random_state=42)
y_train_small = y_train_reg_aligned.iloc[X_train_small.index]

# Train a small Random Forest for testing
rf_reg = RandomForestRegressor(n_estimators=10, max_depth=10, random_state=42, n_jobs=-1)
rf_reg.fit(X_train_small, y_train_small)

# Predict on a small test subset
y_pred_small = rf_reg.predict(X_test_encoded.iloc[:5000])

print("Random Forest trained on small sample successfully!")

Random Forest trained on small sample successfully!


In [3]:
rf_reg_full = RandomForestRegressor(
    n_estimators=100,  # increase later if needed
    max_depth=15,      # adjust for performance
    random_state=42,
    n_jobs=-1
)
rf_reg_full.fit(X_train_encoded, y_train_reg_aligned)
y_pred_full = rf_reg_full.predict(X_test_encoded)

NameError: name 'RandomForestRegressor' is not defined